<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 06: Ejercicios de repaso 2 · Clase 6</p>
  </div>
</div>

# Ejercicios de repaso 2 · Clase 6: Tuning, interpretación y persistencia

Estos ejercicios repasan de manera práctica la **Clase 6**: cómo asegurar que la evaluación sea
**honesta** (fuga de datos y *pipelines*), cómo **seleccionar variables** con método, afinar
hiperparámetros con **tuning**, explicar el modelo con **SHAP** (global y local), **guardarlo** listo
para producción y traducir las probabilidades en una **decisión de negocio rentable**.

Son **actividades formativas**: sirven para practicar, ganar soltura y recibir *feedback*.

**Cómo trabajarlos:**

- Ejecuten primero las celdas de **preparación** y la celda de **datos**.
- En cada ejercicio hay una celda de código con **pistas** como comentarios: casi siempre basta con
  copiar la pista, quitarle el `#` y completarla. El código es corto a propósito (*low-code*).
- Después de cada ejercicio hay una celda **"Su lectura"**: lo central del diplomado no es el código,
  sino **interpretar y decidir**. Respondan ahí con sus palabras.
- Cuando terminen, comparen con el notebook de **solución comentada**: explica por qué se escribió cada
  línea y, en los ejercicios principales, entrega un *prompt* para pedirle a una IA que les ayude a
  corregir su propio código.

## Preparación del entorno

Instalamos las librerías necesarias y dejamos la configuración general de importaciones.

In [ ]:
%%capture
!pip install -q pycaret==4.0.0a8 plotly ipywidgets shap "setuptools<81"

In [ ]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px

# en Colab los datos se leen del repo público; en local, desde ../datasets/
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

## Bloque A · Fuga de datos y evaluación honesta

**Caso:** campaña de depósito a plazo de un banco (`banco_marketing.csv`, datos reales de UCI Bank Marketing).
El banco quiere decidir **a qué clientes llamar** en la próxima campaña. La columna `contrata` vale **1**
si el cliente contrató el depósito y **0** si no.

En este bloque aprendemos a distinguir la evaluación honesta de la ilusión de un modelo con trampa.

In [ ]:
banco = pd.read_csv(BASE + 'banco_marketing.csv')
print('Filas y columnas:', banco.shape)
banco.head()

### Ejercicio A1 · La trampa de la variable que sabe la respuesta

Entrenen un clasificador rápido `rf` (*Random Forest*) que incluya **todas** las columnas de la tabla
(incluida `duracion_llamada_seg`). Observen su métrica **AUC** y la **importancia de variables**.
Luego respondan: ¿por qué este resultado es un engaño?, ¿conocemos la duración de la llamada el próximo
lunes a las 9:00 AM cuando el ejecutivo decide a quién marcar?

**Criterio de éxito:** ver el AUC inflado (> 0.90), la dominancia absoluta de la duración y explicar por qué es una fuga tipo 2.

In [ ]:
# pista:
# from pycaret.tasks import ClassificationExperiment
# exp_trampa = ClassificationExperiment(target='contrata', session_id=42, train_size=0.7, fold=3).fit(banco)
# rf_trampa = exp_trampa.create_model('rf')
# exp_trampa.plot_model(rf_trampa.pipeline, plot='feature')

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio A2 · El experimento honesto y el pipeline como maleta

Descarten la columna `duracion_llamada_seg` de la tabla (`df = banco.drop(columns='duracion_llamada_seg')`).
Creen el experimento honesto con `train_size=0.7`, `fold=5`, `normalize=True` y `fold_strategy='stratifiedkfold'`.
Entrenen un modelo base `rf` y muestren su objeto `.pipeline`. ¿Qué transformaciones viajan dentro de esa "maleta"?
¿Por qué deben ajustarse **solo** con los datos de entrenamiento?

**Criterio de éxito:** el experimento honesto creado sin la fuga, el AUC real (~0.71) y la explicación del pipeline.

In [ ]:
# pista:
# df_honesto = banco.drop(columns='duracion_llamada_seg')
# exp = ClassificationExperiment(target='contrata', session_id=42, train_size=0.7, fold=5,
#                                normalize=True, fold_strategy='stratifiedkfold').fit(df_honesto)
# base = exp.create_model('rf')
# modelo_base = base.pipeline
# print('AUC honesto:', base.metrics['AUC'].values[0])
# modelo_base

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque B · Selección de variables con criterio y con método

Tener muchas columnas no siempre ayuda: variables irrelevantes agregan ruido y encarecen los sistemas.
En este bloque aprendemos a medir el aporte real de cada variable y a preguntarnos **cuántas variables bastan**.

### Ejercicio B1 · Importancia por permutación (PFI): la medida honesta

Calculen la **importancia por permutación** del modelo honesto sobre el conjunto de prueba (`exp.X_test` e
`exp.y_test`) usando `permutation_importance` de scikit-learn con scoring `'roc_auc'`.
¿Cuál es la variable que más pesa en la predicción honesta? ¿Por qué desordenar una columna al azar nos
permite saber si realmente importa?

**Criterio de éxito:** una tabla o serie con la importancia por permutación y la identificación clara de la variable dominante.

In [ ]:
# pista:
# from sklearn.inspection import permutation_importance
# pfi = permutation_importance(modelo_base, exp.X_test, exp.y_test, scoring='roc_auc', n_repeats=5, random_state=42)
# s_pfi = pd.Series(pfi.importances_mean, index=exp.X_test.columns).sort_values(ascending=False)
# s_pfi

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio B2 · ¿Cuántas variables necesita realmente el modelo?

Esta celda **ya viene provista**: ejecuta RFECV (eliminación recursiva de variables con validación
cruzada) para ver cómo evoluciona el AUC a medida que agregamos o quitamos variables.
Ejecútenla y respondan: ¿con cuántas variables se alcanza prácticamente el mejor desempeño?
Si una variable agrega apenas 0,002 de AUC pero cuesta millones capturarla, ¿la conservarían?

**Criterio de éxito:** leer en la tabla cuántas variables bastan y argumentar una decisión equilibrando métricas y costos.

In [ ]:
from sklearn.base import clone
from sklearn.feature_selection import RFECV

# pasamos los datos por el preprocesamiento del pipeline
preproc = modelo_base[:-1]
estimador_puro = clone(modelo_base[-1])
X_train_trans = preproc.transform(exp.X_train)

# rfecv va quitando variables de a una para ver cuándo deja de mejorar el modelo
rfecv = RFECV(estimator=estimador_puro, step=1, cv=3, scoring='roc_auc')
rfecv.fit(X_train_trans, exp.y_train)

res_rfecv = pd.DataFrame({
    'Variables conservadas': range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    'AUC (validación cruzada)': rfecv.cv_results_['mean_test_score'].round(4)
})
display(res_rfecv)
print('Número óptimo según el algoritmo:', rfecv.n_features_)

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque C · Hiperparámetros y tuning

Un **hiperparámetro** es una perilla que fijamos **antes** de entrenar (como la profundidad del árbol
`max_depth` o el número de estimadores `n_estimators`). El *tuning* busca de forma sistemática la mejor
combinación. En este bloque vemos cómo afinar el modelo y cómo evaluar si la mejora es real.

### Ejercicio C1 · Afinar el modelo con tune_model

Apliquen `tune_model` sobre `modelo_base` realizando 15 iteraciones con `optimize='AUC'`.
Muestren los mejores hiperparámetros encontrados (`mejora.best_params`) y exploren las primeras filas
de la tabla de búsqueda `pd.DataFrame(mejora.cv_results)`. ¿Cuánto mejoró el AUC promedio en la validación
cruzada? ¿El tuning transformó el modelo o solo afinó décimas?

**Criterio de éxito:** ejecutar el tuning con optimize='AUC', mostrar los mejores hiperparámetros y cuantificar la mejora en CV.

In [ ]:
# pista:
# mejora = exp.tune_model(modelo_base, n_iter=15, optimize='AUC')
# mejora.best_params
# pd.DataFrame(mejora.cv_results)[['param_max_depth', 'param_n_estimators', 'mean_test_score']].head()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio C2 · El examen único en el holdout: Base vs Afinado

Evalúen `modelo_base` y `mejora.pipeline` sobre el **conjunto de prueba** (`predict_model`).
Pongan lado a lado el AUC de ambos en una tabla. ¿El modelo afinado confirma su ventaja con datos nuevos
que nunca vio? ¿Por qué esta comparación en el holdout se realiza **una sola vez** al final del proceso?

**Criterio de éxito:** una tabla comparativa de AUC en el holdout y la justificación de no reusar el conjunto de prueba.

In [ ]:
# pista:
# res_base = exp.predict_model(modelo_base)
# res_afinado = exp.predict_model(mejora.pipeline)
# comp = pd.DataFrame({
#     'Modelo': ['Base', 'Afinado'],
#     'AUC Holdout': [res_base.metrics['AUC'].iloc[0], res_afinado.metrics['AUC'].iloc[0]]
# })
# comp

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque D · Interpretabilidad con SHAP y persistencia

Los modelos modernos son complejos. Para generar confianza, cumplir con regulaciones (como la Ley 21.719
en Chile) y detectar sesgos, debemos **abrir la caja negra**: explicar qué variables pesan en general
(**global**) y por qué se tomó una decisión sobre una persona en particular (**local**).
Luego, empaquetamos el modelo para que el banco pueda usarlo en producción.

### Ejercicio D1 · Interpretabilidad global con beeswarm de SHAP

Generen el gráfico **beeswarm de SHAP** del modelo ganador usando `exp.plot_model(..., plot='shap_beeswarm')`.
Identifiquen: (a) ¿cuál es la variable más influyente?, (b) ¿cómo afecta tener valores altos de `tasa_euribor`
(color rojo) a la probabilidad de contratar el depósito?

**Criterio de éxito:** generar el gráfico beeswarm e interpretar correctamente el impacto del valor de la tasa de interés.

In [ ]:
# pista: exp.plot_model(mejora.pipeline, plot='shap_beeswarm')

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D2 · Interpretabilidad local: el porqué de un cliente concreto

Esta celda viene provista para calcular la explicación local SHAP del **primer cliente del conjunto de prueba**.
Ejecútenla, miren las fuerzas que empujan la predicción y redacten una explicación de dos líneas en
lenguaje sencillo, como si tuvieran que explicárselo al ejecutivo que llamará a ese cliente.

**Criterio de éxito:** ejecutar la celda provista y traducir las fuerzas SHAP individuales a una explicación comprensible para negocio.

In [ ]:
import shap

# sacamos el modelo del pipeline y preparamos los datos de prueba
pipe = mejora.pipeline
pre, rf_final = pipe[:-1], pipe[-1]
Xt = pd.DataFrame(pre.transform(exp.X_test), columns=pre.get_feature_names_out())

explainer = shap.TreeExplainer(rf_final)
shap_values = explainer(Xt)

# tomamos la explicación para la decisión de contratar (clase 1)
contacto_idx = 0
if len(shap_values.shape) == 3:
    exp_contacto = shap_values[contacto_idx, :, 1]
    base_val = float(shap_values.base_values[contacto_idx, 1])
else:
    exp_contacto = shap_values[contacto_idx]
    base_val = float(shap_values.base_values[contacto_idx])

prob_predicha = pipe.predict_proba(exp.X_test.iloc[[contacto_idx]])[0, 1]

print(f'Cliente #{contacto_idx}:')
print(f'  Probabilidad base promedio del banco: {base_val:.3f}')
print(f'  Probabilidad predicha del cliente   : {prob_predicha:.3f}')

fuerzas = pd.Series(exp_contacto.values, index=Xt.columns).sort_values()
print('\nFuerzas SHAP que empujan la decisión (valores positivos suman probabilidad):')
display(fuerzas.round(4))

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D3 · Persistencia: empaquetar, guardar y predecir en producción

Finalicen el modelo ganador con `exp.finalize_model(...)` (para reentrenar con el 100% de los datos) y
guárdenlo como `'modelo_banco'`. Luego cárguenlo con `exp.load_model(...)` y generen predicciones sobre los
3 clientes nuevos simulados en la celda. ¿Tuvimos que escalar o imputar los datos nuevos a mano?

**Criterio de éxito:** completar el flujo finalize -> save -> load y comprobar que el pipeline predice datos crudos de forma autónoma.

In [ ]:
# pista:
# final = exp.finalize_model(mejora.pipeline)
# exp.save_model(final.pipeline, 'modelo_banco')
# modelo_prod = exp.load_model('modelo_banco')
# nuevos = df_honesto.drop(columns='contrata').head(3)
# modelo_prod.predict(nuevos)

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque E · De la métrica a la decisión de negocio

El objetivo final de un modelo en una empresa no es tener un AUC alto, sino **generar valor o ahorrar dinero**.
En este bloque traducimos la matriz de predicción a pesos chilenos ($).

### Ejercicio E1 · La cuenta económica de la campaña y el umbral óptimo

Supongan las siguientes condiciones de negocio para la campaña:
- Cada depósito captado genera un **margen neto de $60.000**.
- Cada llamada realizada cuesta **$5.000** (tiempo del ejecutivo, telefonía).
Calculen la ganancia neta en el conjunto de prueba (2.400 clientes) bajo tres estrategias:
1. **No hacer nada:** ganancia = $0.
2. **Fuerza bruta (llamar a todos los 2.400 contactos).**
3. **Usar el modelo con umbral óptimo:** llamar solo si la probabilidad predicha es mayor o igual a
   `5.000 / 60.000 ≈ 0.0833` (el punto donde el valor esperado de la llamada cubre su costo).
¿Cuánto dinero extra genera el modelo respecto a la fuerza bruta?

**Criterio de éxito:** calcular la ganancia en pesos de llamar a todos vs usar el modelo y concluir con la cifra de valor aportado.

In [ ]:
# pista:
# y_real = exp.y_test.values
# probs = res_afinado.predictions['prediction_score'].values
# dep_todos = y_real.sum()
# costo_todos = len(y_real) * 5000
# ganancia_todos = dep_todos * 60000 - costo_todos
#
# umbral = 5000 / 60000
# llaman_modelo = probs >= umbral
# dep_modelo = y_real[llaman_modelo].sum()
# costo_modelo = llaman_modelo.sum() * 5000
# ganancia_modelo = dep_modelo * 60000 - costo_modelo

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

---

## Síntesis formativa

En este repaso 2 de la **Clase 6** recorrimos el camino que transforma un prototipo en una herramienta de negocio:

1. **Evaluación honesta:** eliminamos la variable que "sabía la respuesta" (`duracion_llamada_seg`) y
   comprendimos que el *pipeline* protege el preprocesamiento dentro del entrenamiento.
2. **Selección de variables:** medimos el peso real de las columnas con importancia por permutación (PFI) y
   vimos con RFECV cuántas variables necesita realmente el modelo para no encarecer la operación.
3. **Tuning:** afinamos los hiperparámetros con `tune_model(..., optimize='AUC')` y constatamos que el tuning
   afina décimas, requiriendo siempre un examen final único en el *holdout*.
4. **Interpretabilidad:** abrimos la caja negra con SHAP a nivel global (beeswarm) y local (waterfall),
   cumpliendo con exigencias regulatorias y de confianza.
5. **Persistencia y despliegue:** empaquetamos el pipeline completo (`finalize` -> `save` -> `load`) para
   procesar datos crudos en producción.
6. **Valor de negocio:** convertimos las probabilidades en una política de llamadas con umbral económico,
   demostrando que el modelo aporta millones de pesos frente a la fuerza bruta.

## Atribución de datos

**Campaña de depósito a plazo de banco comercial** (`banco_marketing.csv`)

**Creador:** S. Moro, P. Rita y P. Cortez (2014).
**Fuente:** [UCI Machine Learning Repository: Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing).
**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
**Modificación:** selección de 8.000 observaciones, traducción y codificación de columnas ordinales y discretas,
preservando la estructura original para el modelado didáctico.